<a href="https://colab.research.google.com/github/Ibrah-N/PNID_Symbol_Detection_For_Large_Complex_Diagrams/blob/main/Symbol_Detection_RF_DETR_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Dataset

##Utils

In [4]:
import warnings
warnings.filterwarnings("ignore")

### Dataset Verification

In [64]:
###################################################
# script for verifying the dataset
# format of input data
#   - images (.jpg, .png ....)
#   - labels (yolo_format_annotations(cls, x_cen, y_cen, width, height)) # normalized values
#####################################################
import os
import cv2
import random
from google.colab.patches import cv2_imshow


def xywh_to_xyxy(x_center, y_center,
                 bbox_width, bbox_height,
                 img_width, img_height):
  """Converting normalized yolo labels (class_id, x, y, w, h) into unnormalized (class_id, x1, y1, x2, y2)

    Return:
      list (class_id, x1, y1, x2, y2)
  """

  # Denormalize YOLO coords
  x_center *= img_width
  y_center *= img_height
  bbox_width *= img_width
  bbox_height *= img_height

  x1 = int(x_center - bbox_width / 2)
  y1 = int(y_center - bbox_height / 2)
  x2 = int(x_center + bbox_width / 2)
  y2 = int(y_center + bbox_height / 2)

  return [x1, y1, x2, y2]


def draw_bbox(image, lines, line_width=2):
  """Draw bounding box over the image

    Return:
      Image array
  """
  h, w = image.shape[:2]

  for idx_line, line in enumerate(lines):

    parts = line.strip().split()
    if len(parts) != 5:
        continue

    class_id, x_center, y_center, bbox_width, bbox_height = map(float, parts)
    class_id, x1, y1, x2, y2 = int(class_id), *xywh_to_xyxy(x_center, y_center,
                                                            bbox_width, bbox_height,
                                                            w, h)

    # Pick color for this class
    color = (random.randint(0, 255),
            random.randint(0, 255),
            random.randint(0, 255))
    global colors # defined in __name__=="__main__" memory block

    if class_id not in colors.keys():

      while True:
        # prevent duplicate colors for multiple classes
        if color in colors.values():
            color = (random.randint(0, 255),
                    random.randint(0, 255),
                    random.randint(0, 255))
            continue

        else:
          # add new color for new class
          colors[class_id] = color
          break


    color = colors[class_id]

    # Draw rectangle + class ID
    cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
    cv2.putText(image, str(class_id), (x1, max(20, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

  return image


def main(image_path, label_path, image_save_path):
  """Main Code looping images patha and labels path; saving resltand images

  """

  # read image
  image = cv2.imread(image_path)

  if image is None:
    print(f"Could not read image: {image_path}")
    return

  # read labels
  with open(label_path, 'r') as label_file:
    lines = label_file.readlines()

  image = draw_bbox(image, lines)

  # save image
  cv2.imwrite(image_save_path, image)
  print("Image {} Saved to {}".format(image_path, image_save_path))


### Dataset Classes Analytics & Jesnofication

In [ ]:
import os
import cv2
import yaml
import json


###################################################
# script for verifying the dataset
# format of input data
#   - images (.jpg, .png ....)
#   - labels (yolo_format_annotations(cls, x_cen, y_cen, width, height)) # normalized values
#####################################################
import os
import cv2
import random
from google.colab.patches import cv2_imshow


def update_class_count(class_id):
  pass

def read_yaml_file(yaml_path):
  """Read yaml file and return the dict(Keys=class_id: Value=class_name)

    Return:
      dict
  """
  with open(yaml_path, 'r') as yaml_file:
    yaml_data = yaml.safe_load(yaml_file)

  return yaml_data['names']


def parse_labels(image, lines, line_width=2):
  """Parse Yolo Data

    Return:
      Image array
  """
  h, w = image.shape[:2]

  for idx_line, line in enumerate(lines):

    parts = line.strip().split()
    if len(parts) != 5:
        continue

    class_id, box = int(parts[0]), parts[1:]



def main(image_path, label_path, image_save_path):
  """Main Code looping images patha and labels path; saving resltand images

  """

  # read image
  image = cv2.imread(image_path)

  if image is None:
    print(f"Could not read image: {image_path}")
    return

  # read labels
  with open(label_path, 'r') as label_file:
    lines = label_file.readlines()

  image = draw_bbox(image, lines)

  # save image
  cv2.imwrite(image_save_path, image)
  print("Image {} Saved to {}".format(image_path, image_save_path))


## Unzip From Drive

In [1]:
from google.colab import drive
drive.mount("Ibrah")

Mounted at Ibrah


In [10]:
import os

zip_data_path = "/content/Ibrah/MyDrive/Symbol_Detection/Classification/Classification_Datasets/Roboflow Symbols Batch.zip"

!unzip -q {zip_data_path.replace(" ", "\ ")} -d /content/

##Unzip Sub Batches


In [62]:
import os

# -- dataset paths --
list_paths = ["/content/Roboflow Symbols Batch/First Batch.v8i.yolov8.zip", "/content/Roboflow Symbols Batch/Second_batch.v10i.yolov8.zip",
              "/content/Roboflow Symbols Batch/Third Batch Z1.v9i.yolov8.zip", "/content/Roboflow Symbols Batch/Third Batch Z2.v8i.yolov8.zip",
              "/content/Roboflow Symbols Batch/Third Batch.v8i.yolov8.zip", "/content/Roboflow Symbols Batch/batch-4-raw_annotated_dataset.v8i.yolov8.zip",
              "/content/Roboflow Symbols Batch/Batch 5 - raw_annotated_dataset.v8i.yolov8.zip", "/content/Roboflow Symbols Batch/Batch 6 - raw_annotated_dataset.v8i.yolov8.zip",
              "/content/Roboflow Symbols Batch/Batch 7- raw_annotated_dataset_05.v8i.yolov8.zip", "/content/Roboflow Symbols Batch/batch_8_raw_annotated_dataset.v7i.yolov8.zip"]

# -- save paths for each zip --
save_paths = ["dataset_01", "dataset_02", "dataset_03_01", "dataset_03_02",
              "dataset_03_03", "dataset_04", "dataset_05", "dataset_06",
              "dataset_07", "dataset_08"]


save_base_path = "/content/combine_symbol_v0"

os.makedirs(save_base_path, exist_ok=True)


for idx in range(len((list_paths))):
  print("Unziping {} to {}".format(list_paths[idx], os.path.join(save_base_path, save_paths[idx])))

  !unzip -q {list_paths[idx].replace(" ", "\ ")} -d {os.path.join(save_base_path, save_paths[idx])}

Unziping /content/Roboflow Symbols Batch/First Batch.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_01
Unziping /content/Roboflow Symbols Batch/Second_batch.v10i.yolov8.zip to /content/combine_symbol_v0/dataset_02
Unziping /content/Roboflow Symbols Batch/Third Batch Z1.v9i.yolov8.zip to /content/combine_symbol_v0/dataset_03_01
Unziping /content/Roboflow Symbols Batch/Third Batch Z2.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_03_02
Unziping /content/Roboflow Symbols Batch/Third Batch.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_03_03
Unziping /content/Roboflow Symbols Batch/batch-4-raw_annotated_dataset.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_04
Unziping /content/Roboflow Symbols Batch/Batch 5 - raw_annotated_dataset.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_05
Unziping /content/Roboflow Symbols Batch/Batch 6 - raw_annotated_dataset.v8i.yolov8.zip to /content/combine_symbol_v0/dataset_06
Unziping /content/Roboflow Symbols Batch/Batch 7- raw

### Dataset Verification

In [66]:
###################################################
# REFERENCE: Functions defined in Dataset Utils
#
# script for verifying the dataset
# format of input data
#   - images (.jpg, .png ....)
#   - labels (yolo_format_annotations(cls, x_cen, y_cen, width, height)) # normalized values
#####################################################

if __name__ == "__main__":
  images_path = "/content/combine_symbol_v0/dataset_01/train/images"
  labels_path = "/content/combine_symbol_v0/dataset_01/train/labels"
  save_path = "/content/tests_saved"
  samples = 20 # number of samples to be used
  patch_size = 1792  # Not directly used unless you want to draw grid or check sizes

  os.makedirs(save_path, exist_ok=True)


  for img_idx, img_file in enumerate(os.listdir(images_path)):

    if not img_file.lower().endswith(('.jpg', '.png', '.jpeg')):
        continue

    image_path = os.path.join(images_path, img_file)
    label_path = os.path.join(labels_path, img_file.rsplit('.', 1)[0] + ".txt")

    image_save_path = os.path.join(save_path, img_file)

    main(image_path, label_path, image_save_path)


    if img_idx >= samples:
      break

Image /content/combine_symbol_v0/dataset_01/train/images/P14456A-14-01-08-0686-1_patch_2_0_jpg.rf.c1ed2c568d445d76e14b37693fba9ed1.jpg Saved to /content/tests_saved/P14456A-14-01-08-0686-1_patch_2_0_jpg.rf.c1ed2c568d445d76e14b37693fba9ed1.jpg
Image /content/combine_symbol_v0/dataset_01/train/images/P14456A-14-01-08-0668-1_patch_3_0_jpg.rf.62ba977216c3092cce5004a0a7054d46.jpg Saved to /content/tests_saved/P14456A-14-01-08-0668-1_patch_3_0_jpg.rf.62ba977216c3092cce5004a0a7054d46.jpg
Image /content/combine_symbol_v0/dataset_01/train/images/P14456A-14-01-08-0694-1_patch_0_4_jpg.rf.b4dcd6b45509df82a6536788e3086d73.jpg Saved to /content/tests_saved/P14456A-14-01-08-0694-1_patch_0_4_jpg.rf.b4dcd6b45509df82a6536788e3086d73.jpg
Image /content/combine_symbol_v0/dataset_01/train/images/P14456A-14-01-08-0663-1_patch_3_1_jpg.rf.936c00918c2953cc4ce7d98911f75c84.jpg Saved to /content/tests_saved/P14456A-14-01-08-0663-1_patch_3_1_jpg.rf.936c00918c2953cc4ce7d98911f75c84.jpg
Image /content/combine_symbo

### Classes Analytic & Jsonification

In [ ]:
###################################################
# REFERENCE: Functions defined in Dataset Utils
#
# script for verifying the dataset
# format of input data
#   - images (.jpg, .png ....)
#   - labels (yolo_format_annotations(cls, x_cen, y_cen, width, height)) # normalized values
#####################################################
